# CASMI Historical Hybrid + Neural Routing
Frozen training-only metadata MLP. Preserve historical full ranking at confidence >=0.5; otherwise neural RRF weight 0.75, with first-place replacement allowed. Offline CPU inference. Local validation is not a leaderboard score.


In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, subprocess, sys, zipfile
roots = list(Path('/kaggle/input').rglob('recipe.json'))
if len(roots) == 1:
    bundle = roots[0].parent
else:
    archives = list(Path('/kaggle/input').rglob('casmi_secondary_bundle.zip'))
    assert len(archives) == 1, f'Expected one model archive, got {archives}'
    bundle = Path('/kaggle/working/casmi_runtime')
    bundle.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(bundle)
for name, expected in json.loads((bundle / 'SHA256SUMS.json').read_text()).items():
    assert hashlib.sha256((bundle / name).read_bytes()).hexdigest() == expected, name
required = json.loads((bundle / 'runtime_versions.json').read_text())
try:
    installed = importlib.metadata.version('rdkit')
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != required['rdkit']:
    wheels = list((bundle / 'wheels').glob('rdkit*.whl'))
    assert len(wheels) == 1
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', str(wheels[0])])
sys.path.insert(0, str(bundle))
from casmi_ml.secondary_inference import predict
predict(bundle / 'recipe.json', '/kaggle/input/competition',
        '/kaggle/input/coconut_structures.parquet', '/kaggle/working/submission.csv')
